In [ ]:
import os
import json
from docx import Document
import re

def bio_tag(doc_name, offset=3):
    """
    Генерация биотэгов для NER задачи извечения ссылок
    """
    tags = []
    document = Document(doc_name)

    
    for paragraph in document.paragraphs:
        if hasattr(paragraph, "hyperlinks") and paragraph.hyperlinks:  
            for link in paragraph.hyperlinks:
                
                words = re.findall(r"\b[\w'.-]+\b|[(),.!?:;]", paragraph.text)
                link_words = re.findall(r"\b[\w'.-]+\b|[(),.!?:;]", link.text)
                
                for word in words:
                    if word in link_words:
                        
                        tags.append([word, "I-LINK"])
                    else:
                        
                        tags.append([word, "O"])
        else:
            
            words = re.findall(r"\b[\w'.-]+\b|[(),.!?:;]", paragraph.text)
            for word in words:
                tags.append([word, "O"])

    
    for i, tag in enumerate(tags):
        if tag[1] == "I-LINK":
            
            if i == 0 or tags[i - 1][1] not in ("B-LINK", "I-LINK"):
                tags[i][1] = "B-LINK"

    
    new_tags = []
    for word, tag in tags:
        if tag == "I-LINK":
            
            parts = re.findall(r"\d+|\D", word)
            for part in parts:
                if part.strip():  
                    new_tags.append([part, tag])
        else:
            
            new_tags.append([word, tag])

    return new_tags

def process_folder(folder_path, output_json_path):
    all_tags = []

    
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            file_path = os.path.join(folder_path, filename)
            tags = bio_tag(file_path)
            all_tags.extend(tags)  

    
    with open(output_json_path, 'w', encoding='utf-8') as json_file:
        json.dump(all_tags, json_file, ensure_ascii=False, indent=4)

    print(f"Bio tags have been saved to {output_json_path}")


current_folder = os.getcwd()


output_json_path = os.path.join(current_folder, "bio_tags.json")


process_folder(current_folder, output_json_path)